# Finetune Pipeline: ViT / DeiT / CaiT / BEiT
Bu notebook `finetune/train_models.py` dosyasından alınan kodu blok-blok hâline getirir.
Her blok üstünde kısa bir açıklama (başlık) ve ardından ilgili kod hücresi bulunmaktadır.
Kullanım: önce `Configuration` hücresini kendi veri yolunuza göre güncelleyin, sonra hücreleri sırayla çalıştırın.

## 1 — Imports
Gerekli kütüphaneler ve yardımcı araçlar burada import edilir.

In [1]:
# Imports
import os
import json
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision import datasets

import timm
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm


c:\Users\emirh\anaconda3\envs\pytorch-v1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2 — Configuration
Bu hücrede eğitim için kullanılacak sabit (gömülü) parametreler bulunmaktadır.
İstediğiniz değişiklikleri burada yapın; script komut satırı argümanları istemeyecek şekilde gömülüdür.

In [2]:
# Configuration (gömülü)
data_dir = r'C:/Users/emirh/Desktop/Projects/datasets/input_sk'  # Update if needed
#models = ['vit_small_patch16_224', 'deit_small_patch16_224', 'cait_xxs36_224', 'beit_base_patch16_224', 'swin_small_patch4_window7_224', 'pvt_v2_b0', 'convit_tiny']
# Training: include MobileViT and MaxViT (using timm model keys)
# Example timm keys: 'mobilevit_xs', 'mobilevit_s', 'maxvit_tiny_rw_224', 'maxvit_small_tf_224'
models = ['mobilevit_xs', 'maxvit_tiny_rw_224']
image_size = 224
batch_size = 32
num_workers = 4
epochs = 50
lr = 1e-4
weight_decay = 1e-4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pretrained = True
reduce_lr_patience = 4
early_stopping_patience = 10

print('Using device:', device)


Using device: cuda


## 3 — Data Loaders
`get_dataloaders` fonksiyonu ImageFolder formatındaki veri kümesini yükler ve DataLoader döndürür.

In [3]:
def get_dataloaders(data_dir, image_size=224, batch_size=32, num_workers=4):
    train_dir = os.path.join(data_dir, 'train')
    val_dir = os.path.join(data_dir, 'val')
    test_dir = os.path.join(data_dir, 'test')

    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]

    train_transforms = T.Compose([
        T.RandomResizedCrop(image_size),
        T.RandomHorizontalFlip(),
        T.ColorJitter(0.1, 0.1, 0.1, 0.1),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    val_transforms = T.Compose([
        T.Resize(int(image_size * 1.14)),
        T.CenterCrop(image_size),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])

    if not os.path.isdir(train_dir) or not os.path.isdir(val_dir):
        raise FileNotFoundError(f"Expected dataset with 'train' and 'val' folders under {data_dir}")

    train_ds = datasets.ImageFolder(train_dir, transform=train_transforms)
    val_ds = datasets.ImageFolder(val_dir, transform=val_transforms)
    test_ds = datasets.ImageFolder(test_dir, transform=val_transforms) if os.path.isdir(test_dir) else None

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True) if test_ds else None

    class_names = train_ds.classes
    num_classes = len(class_names)

    return {'train': train_loader, 'val': val_loader, 'test': test_loader}, {'train': len(train_ds), 'val': len(val_ds), 'test': len(test_ds) if test_ds else 0}, class_names

## 4 — Model creation
`create_model` fonksiyonu `timm.create_model` ile ön-eğitimli modeli yükler ve sınıflandırma başlığını (`head` / `fc` / `classifier`) uyarlamaya çalışır.

In [4]:
def create_model(model_name, num_classes, pretrained=True, device='cuda'):
    # Check available timm model names first and give helpful suggestions on error
    try:
        available = timm.list_models()
    except Exception:
        available = []

    if model_name not in available:
        import difflib
        close = difflib.get_close_matches(model_name, available, n=6)
        raise RuntimeError(
            f"Unknown model '{model_name}'. Available models count={len(available)}. "
            f"Did you mean one of: {close}?\nCall `timm.list_models()` to list available model names."
        )

    try:
        model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    except Exception as e:
        print(f"Model construction with num_classes failed for {model_name}: {e}. Attempting manual head replacement.")
        model = timm.create_model(model_name, pretrained=pretrained)
        # try to replace common head attributes
        if hasattr(model, 'head') and hasattr(model.head, 'in_features'):
            in_f = model.head.in_features
            model.head = nn.Linear(in_f, num_classes)
        elif hasattr(model, 'fc') and hasattr(model.fc, 'in_features'):
            in_f = model.fc.in_features
            model.fc = nn.Linear(in_f, num_classes)
        elif hasattr(model, 'classifier') and hasattr(model.classifier, 'in_features'):
            in_f = model.classifier.in_features
            model.classifier = nn.Linear(in_f, num_classes)
        else:
            raise RuntimeError(f"Couldn't replace classifier head for {model_name}")
    return model.to(device)


## 5 — Training helpers
`train_one_epoch` ve `evaluate` fonksiyonları eğitim ve değerlendirme döngülerini uygular.

In [5]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm(loader, leave=False)
    for images, targets in pbar:
        images = images.to(device)
        targets = targets.to(device)
        outputs = model(images)
        loss = criterion(outputs, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == targets).sum().item()
        total += images.size(0)
        pbar.set_description(f"Train loss {loss.item():.4f}")

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds_all = []
    labels_all = []
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            preds_all.extend(preds.cpu().numpy().tolist())
            labels_all.extend(targets.cpu().numpy().tolist())

    total = len(labels_all)
    epoch_loss = running_loss / total if total > 0 else 0.0
    acc = accuracy_score(labels_all, preds_all) if total > 0 else 0.0
    prec = precision_score(labels_all, preds_all, average='macro', zero_division=0) if total > 0 else 0.0
    rec = recall_score(labels_all, preds_all, average='macro', zero_division=0) if total > 0 else 0.0
    f1 = f1_score(labels_all, preds_all, average='macro', zero_division=0) if total > 0 else 0.0
    cm = confusion_matrix(labels_all, preds_all) if total > 0 else None
    return epoch_loss, acc, prec, rec, f1, cm

## 6 — Plotting and saving results
Grafikler (loss/accuracy) ve karışıklık matrisi oluşturulur ve `results/<model_name>/` dizinine kaydedilir.

In [6]:
def plot_and_save(history, cm, class_names, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    # loss/acc
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], label='Train Loss')
    plt.plot(epochs, history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Loss')

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], label='Train Acc')
    plt.plot(epochs, history['val_acc'], label='Val Acc')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('Accuracy')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'loss_acc.png'))
    plt.close()

    if cm is not None:
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
        plt.ylabel('True')
        plt.xlabel('Predicted')
        plt.title('Confusion Matrix')
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, 'confusion_matrix.png'))
        plt.close()

## 7 — Train single model (core training loop)
`train_model` fonksiyonu bir model için eğitim döngüsünü, ReduceLROnPlateau ve erken durdurmayı uygular.

In [ ]:
def train_model(data_dir, model_name, output_root='results', image_size=224, batch_size=32, epochs=10, lr=1e-4, weight_decay=1e-4, device='cuda', num_workers=4, pretrained=True, reduce_lr_patience=4, early_stopping_patience=10):
    loaders, sizes, class_names = get_dataloaders(data_dir, image_size=image_size, batch_size=batch_size, num_workers=num_workers)
    num_classes = len(class_names)
    model = create_model(model_name, num_classes=num_classes, pretrained=pretrained, device=device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=reduce_lr_patience)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_prec': [], 'val_rec': [], 'val_f1': [], 'epoch_times': []}

    best_val_loss = float('inf')
    best_f1 = -1.0
    best_state = None
    no_improve_epochs = 0
    out_dir = os.path.join(output_root, model_name)
    os.makedirs(out_dir, exist_ok=True)

    train_start_time_all = time.time()
    cm = None
    for epoch in range(1, epochs + 1):
        epoch_start = time.time()
        train_loss, train_acc = train_one_epoch(model, loaders['train'], criterion, optimizer, device)
        val_loss, val_acc, val_prec, val_rec, val_f1, cm = evaluate(model, loaders['val'], criterion, device)
        # Step scheduler with validation loss
        try:
            scheduler.step(val_loss)
        except Exception:
            pass

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_prec'].append(val_prec)
        history['val_rec'].append(val_rec)
        history['val_f1'].append(val_f1)

        epoch_time = time.time() - epoch_start
        history['epoch_times'].append(epoch_time)

        elapsed = epoch_time
        print(f"{model_name} Epoch {epoch}/{epochs}  train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}  ({elapsed:.1f}s)")

        # save best by val_loss (lower is better)
        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            no_improve_epochs = 0
            best_state = model.state_dict()
            torch.save({'model_state_dict': best_state, 'classes': class_names}, os.path.join(out_dir, f"{model_name}_finetuned_best.pth"))
            print(f"\tValidation loss improved; saved best model (val_loss={best_val_loss:.4f})")
        else:
            no_improve_epochs += 1
            print(f"\tNo improvement for {no_improve_epochs}/{early_stopping_patience} epochs")

        # track best f1 as well
        if val_f1 > best_f1:
            best_f1 = val_f1

        if no_improve_epochs >= early_stopping_patience:
            print('Early stopping triggered')
            break

    train_end_time_all = time.time()
    # save history
    with open(os.path.join(out_dir, 'history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    # plot and save final confusion matrix (using last cm if available)
    plot_and_save(history, cm, class_names, out_dir)

    # compute training summary
    epochs_trained = len(history['epoch_times'])
    total_time = sum(history['epoch_times'])
    avg_epoch_time = total_time / epochs_trained if epochs_trained > 0 else 0.0
    param_count = sum(p.numel() for p in model.parameters())

    training_summary = {
        'model': model_name,
        'requested_epochs': epochs,
        'epochs_trained': epochs_trained,
        'early_stopped': epochs_trained < epochs,
        'total_training_time_sec': total_time,
        'avg_epoch_time_sec': avg_epoch_time,
        'per_epoch_times_sec': history['epoch_times'],
        'num_parameters': int(param_count),
        'num_parameters_millions': round(param_count / 1e6, 3),
        'best_val_loss': best_val_loss,
        'best_val_f1': best_f1,
        'training_start_time': train_start_time_all,
        'training_end_time': train_end_time_all,
        'out_dir': out_dir
    }

    with open(os.path.join(out_dir, 'training_summary.json'), 'w') as f:
        json.dump(training_summary, f, indent=2)

    print(f"Done training {model_name}. Best val_loss={best_val_loss:.4f} best_val_f1={best_f1:.4f}. Results saved to {out_dir}")
    return training_summary


## 8 — Run multiple models (helper)
`run_all` fonksiyonu model listesini iter ve her biri için `train_model` çağırır.

In [ ]:
def run_all(data_dir, models, **kwargs):
    os.makedirs('results', exist_ok=True)
    results = []
    for m in models:
        try:
            r = train_model(data_dir, m, **kwargs)
            results.append(r)
        except Exception as e:
            print(f"Error training {m}: {e}")
    # Do not save a global summary file (summary.json) per user request
    print('All done.')
    return results


## 9 — Run training (execute when ready)
Bu hücreyi çalıştırarak tüm modeller için eğitim sürecini başlatabilirsiniz.
Dikkat: Eğitimi başlatmadan önce `data_dir` içeriğinin doğru olduğundan emin olun.

In [9]:
# Run training for all models (uncomment to run)
# Note: this will execute training sequentially for each model in `models`.
# run_all(data_dir, models, image_size=image_size, batch_size=batch_size, epochs=epochs, lr=lr, weight_decay=weight_decay, device=device, num_workers=num_workers, pretrained=pretrained, reduce_lr_patience=reduce_lr_patience, early_stopping_patience=early_stopping_patience)

---
### Notlar
- Eğitim sırasında GPU kullanımı için `device` değeri otomatik algılanır.
- `data_dir` yolunu gerektiği gibi güncelleyin.
- Eğer tek bir modeli çalıştırmak isterseniz `train_model(...)` fonksiyonunu doğrudan çağırabilirsiniz.

## 10 — Execute training (call methods)
Bu hücre, daha önce tanımlanmış `run_all` ve `train_model` fonksiyonlarını çağırmak için örnek kullanım sağlar.
Varsayılan olarak hiçbir şey çalıştırılmaz — eğitim başlatmak için `RUN_ALL` veya `RUN_SINGLE` bayraklarını True yapın.


In [10]:
# Run training for all models (set flags below to actually execute)
# WARNING: Running will start potentially long GPU training sessions.
RUN_ALL = True  # set to True to run all models sequentially
RUN_SINGLE = False  # set to True to run a single model
SINGLE_MODEL_INDEX = 3  # index in `models` list to run when RUN_SINGLE is True

if RUN_ALL:
    run_all(data_dir, models, image_size=image_size, batch_size=batch_size, epochs=epochs, lr=lr, weight_decay=weight_decay, device=device, num_workers=num_workers, pretrained=pretrained, reduce_lr_patience=reduce_lr_patience, early_stopping_patience=early_stopping_patience)
elif RUN_SINGLE:
    m = models[SINGLE_MODEL_INDEX]
    train_model(data_dir, m, output_root='results', image_size=image_size, batch_size=batch_size, epochs=epochs, lr=lr, weight_decay=weight_decay, device=device, num_workers=num_workers, pretrained=pretrained, reduce_lr_patience=reduce_lr_patience, early_stopping_patience=early_stopping_patience)
else:
    print('No training executed. Set RUN_ALL or RUN_SINGLE flags to True to start training.')


c:\Users\emirh\anaconda3\envs\pytorch-v1\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\emirh\.cache\huggingface\hub\models--timm--mobilevit_xs.cvnets_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


mobilevit_xs Epoch 1/50  train_loss=1.3131 val_loss=0.9210 val_acc=0.6892 val_f1=0.2825  (126.6s)
	Validation loss improved; saved best model (val_loss=0.9210)


mobilevit_xs Epoch 2/50  train_loss=0.9427 val_loss=0.8126 val_acc=0.7081 val_f1=0.3166  (127.9s)
	Validation loss improved; saved best model (val_loss=0.8126)


mobilevit_xs Epoch 3/50  train_loss=0.8460 val_loss=0.7233 val_acc=0.7464 val_f1=0.4657  (134.7s)
	Validation loss improved; saved best model (val_loss=0.7233)


mobilevit_xs Epoch 4/50  train_loss=0.7892 val_loss=0.6895 val_acc=0.7532 val_f1=0.4988  (135.9s)
	Validation loss improved; saved best model (val_loss=0.6895)


...

maxvit_tiny_rw_224 Epoch 25/50  train_loss=0.0882 val_loss=0.4621 val_acc=0.9013 val_f1=0.8686  (269.9s)
	No improvement for 6/10 epochs


maxvit_tiny_rw_224 Epoch 26/50  train_loss=0.0790 val_loss=0.4576 val_acc=0.9064 val_f1=0.8659  (269.9s)
	No improvement for 7/10 epochs


maxvit_tiny_rw_224 Epoch 27/50  train_loss=0.0783 val_loss=0.4330 val_acc=0.9084 val_f1=0.8616  (269.9s)
	No improvement for 8/10 epochs


maxvit_tiny_rw_224 Epoch 28/50  train_loss=0.0823 val_loss=0.4613 val_acc=0.9044 val_f1=0.8664  (270.2s)
	No improvement for 9/10 epochs


maxvit_tiny_rw_224 Epoch 29/50  train_loss=0.0824 val_loss=0.4685 val_acc=0.9064 val_f1=0.8679  (270.1s)
	No improvement for 10/10 epochs
Early stopping triggered
Done training maxvit_tiny_rw_224. Best val_loss=0.4181 best_val_f1=0.8686. Results saved to results\maxvit_tiny_rw_224
All done. Summary saved to results/summary.json
